# JEElS 2026 Workshop — Getting Started with HyperSpy & exspy

This notebook introduces the basics of working with multi-dimensional electron
microscopy data using [HyperSpy](https://hyperspy.org/) and
[exspy](https://github.com/userexternal/exspy).

We will create a synthetic multi-dimensional `Signal2D` dataset — a 4D-STEM-like
scan where each probe position records a diffraction pattern — and explore it
interactively.

In [ ]:
%matplotlib ipympl
import numpy as np
import matplotlib.pyplot as plt
import hyperspy.api as hs

hs.print_info()

---
## Generate a synthetic multi-dimensional Signal2D

We simulate a 5 × 5 grid of diffraction patterns (64 × 64 pixels each).
The patterns contain two Gaussian spots whose positions drift smoothly
across the field of view — mimicking a simple crystalline sample with a
gradual bend or strain gradient.

In [ ]:
# Signal dimensions
sx, sy = 64, 64

# Navigation dimensions (scan grid)
nx, ny = 5, 5

# Coordinate grids for the diffraction plane
Y, X = np.mgrid[-sx//2:sx//2, -sy//2:sy//2]

def gaussian_spot(center_x, center_y, sigma=4):
    """2D Gaussian centred at (center_x, center_y)."""
    return np.exp(-((X - center_x)**2 + (Y - center_y)**2) / (2 * sigma**2))

# Build a navigation- (scan-) dependent stack of diffraction patterns
data = np.zeros((nx, ny, sx, sy))

for i in range(nx):
    for j in range(ny):
        # Spots drift linearly with scan position
        dx = -8 + 4 * i
        dy = -6 + 3 * j
        data[i, j] = (
            gaussian_spot(dx, dy, sigma=3)
            + 0.7 * gaussian_spot(-dx * 0.5, -dy * 0.5, sigma=5)
            + 0.05 * np.random.random((sx, sy))  # Poisson-ish background
        )

s = hs.signals.Signal2D(data)
s

The signal is displayed above. Notice that HyperSpy shows the **navigation**
summary (shape `(5, 5)`) and the **signal** dimensions (shape `(64, 64)`).

You can click or drag in the navigator window (the small thumbnail on the
right) to browse individual diffraction patterns interactively.

In [ ]:
# Plot with the HyperSpy interactive navigator
s.plot()

---
## Basic analysis

We can collapse navigation dimensions to form a **virtual image** by summing
various regions of interest.

In [ ]:
# Sum over all diffraction pixels → bright-field image
bf = s.sum(signal_axes=True)
bf.plot()

In [ ]:
# Sum over the entire navigation grid → average diffraction pattern
adp = s.sum(("x", "y"))
adp.plot()

---
## Signal transformations

HyperSpy supports many signal-processing operations that work across all
navigation dimensions automatically.

In [ ]:
# Subtract the average diffraction pattern (centre) from every pixel
s_centre = s / adp

# Plot the centre-corrected version — spot contrasts should be enhanced
s_centre.plot()

In [ ]:
# Take a line profile through the centre of the *average* diffraction pattern
line = adp.interactive_profile(line_plot=True)
line

---
## What next?

- Browse the [HyperSpy user guide](https://hyperspy.org/hyperspy-doc/current/user_guide/index.html)
  for in-depth tutorials.
- Check the [exspy documentation](https://userexternal.github.io/exspy/) for
  electron-microscopy-specific routines (EDS, EELS, …).
- Experiment with the interactive widgets provided by
  `hyperspy_gui_ipywidgets`.